<a href="https://colab.research.google.com/github/Spring-anne/Proyecto-Tecnolochicas-Python/blob/exploracion/ProyectoFinal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ***Análisis de Tendencias de Videojuegos (1980-2025)***
Este proyecto tiene como fin analizar las tendencias de videojuegos populares desde 1980 hasta 2025, con el propósito de identificar cómo ha ido evolucionado la industria a lo largo de estas cuatro décadas.
El problema de investigación se centra en comprender cuáles han sido los factores que han influido en la popularidad de los videojuegos a lo largo del tiempo, tales como los géneros predominantes, las plataformas más utilizadas, las compañías desarrolladoras y distribuidoras, así como la recepción de los usuarios y la crítica especializada.


# ***Imports iniciales***
Importamos las liberias que ocuparemos durante el desarrollo del proyecto

In [57]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import ast

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# ***Exploracion Inicial de datos***
Divagamos un poco en el conjunto de datos cargado usando funciones utiles de pandas



In [ ]:
#Funcion para cargar el archivo csv
def cargar_datos(ruta_csv: str):
    try:
        df = pd.read_csv(ruta_csv)
        print(f"Archivo cargado correctamente: {df.shape[0]} filas, {df.shape[1]} columnas.")
        return df
    except FileNotFoundError:
        raise FileNotFoundError(f"No se encontro el archivo: {ruta_csv}")

#Cargamos el csv y mostramos las primeras 5 filas del conjunto
df_crudo = cargar_datos("backloggd_games.csv")
df_crudo.head(5)

In [ ]:
#Mostramos la cola del conjunto de datos, los ultimos 5 datos
df_crudo.tail(5)

In [62]:
#Elegimos 10 filas aleatorias
df_crudo.sample(10)

,Unnamed: 0,Title,Release_Date,Developers,Summary,Platforms,Genres,Rating,Plays,Playing,Backlogs,Wishlist,Lists,Reviews
11597,11597,JB: The Super Bass,"Dec 15, 1995","['Gaps', 'Naxat Soft']",For some reason the Japanese love bass fishing...,['Super Famicom'],['Sport'],NaN,1,0,0,0,0,0
23516,23516,Lune,"Jun 20, 2016",[],"Lune is a experiential haiku, a short poem ded...",['Windows PC'],"['Indie', 'Simulator']",NaN,0,0,0,0,1,0
20855,20855,Dezaemon Plus,"May 24, 1996",['Athena'],A vertical shooter that lets you create your o...,['PlayStation'],['Shooter'],NaN,2,1,2,3,4,0
2231,2231,Rock Band 3,"Oct 26, 2010","['MTV Games', 'Harmonix Music Systems']",Rock Band 3 adds 83 new songs to the Rock Band...,"['Xbox 360', 'PlayStation 3', 'Wii']",['Music'],4.0,1.2K,5,46,40,131,37
55544,55544,ReCaptcha V4,"Dec 31, 2021",['Performave'],A parody ReCaptcha in which you play chess aga...,['Web browser'],['Card & Board Game'],NaN,2,0,0,0,0,0
50357,50357,Q*bert,"Dec 31, 1983",['Parker Brothers'],NaN,['Handheld Electronic LCD'],[],NaN,1,0,0,0,0,1
2974,2974,J-Stars Victory Vs+,"Jun 24, 2015","['Spike ChunSoft', 'Bandai Namco Entertainment']",Your favorite Shonen Jump heroes jump off the ...,"['PlayStation 4', 'PlayStation 3', 'PlayStatio...","['Brawler', 'Fighting']",2.6,773,4,70,33,68,37
48196,48196,Powerpuff Girls: Zom-B-Gone!,"Oct 11, 2007",[],"Shoot magic elements, watch out for magic bags...",['Web browser'],"['Arcade', 'Point-and-Click']",1.9,12,0,0,0,2,0
6738,6738,Edge,"Dec 01, 2008","['Mobigame', 'Two Tribes B.V.']",In Edge players take direct control of the cub...,"['Windows PC', 'Android', 'Mac', 'Wii U', 'Lin...","['Adventure', 'Indie', 'Platform', 'Puzzle']",3.2,320,3,65,13,48,16
8195,8195,Hentai Nazi,"Jan 07, 2020",['AmagSwag Games'],"Hentai Nazi - It’s not just for you, it smells...",['Windows PC'],['Shooter'],2.3,75,0,4,4,6,13


In [ ]:
#Imprimimos la informacion del df crudo
print(df_crudo.info())

In [53]:
#Exploracion de datos nulos, repetidos
print("--VALORES NULOS--")
print(df_crudo.isnull().sum())
print()
print("--FILAS DUPLICADOS--")
print(df_crudo.duplicated().sum())
print()
print("--TITULOS DUPLICADOS(REMASTERS/RERELEASES/MULTI-PLATAFORMA)--")
print(df_crudo.duplicated(subset=['Title']).sum())

--VALORES NULOS--
Unnamed: 0          0
Title               0
Release_Date        0
Developers          0
Summary          4954
Platforms           0
Genres              0
Rating          34595
Plays               0
Playing             0
Backlogs            0
Wishlist            0
Lists               0
Reviews             0
dtype: int64

--FILAS DUPLICADOS--
0

--TITULOS DUPLICADOS(REMASTERS/RERELEASES/MULTI-PLATAFORMA)--
19015


In [56]:
#Consulta de estadisticas
print("--ESTADISTICAS(RATING)")
print(df_crudo['Rating'].describe())
print("\n")
print("--ESTADISTICAS(RELEASE DATE)")
print(df_crudo['Release_Date'].describe())

--ESTADISTICAS(RATING)
count    25405.000000
mean         3.033171
std          0.735573
min          0.300000
25%          2.600000
50%          3.100000
75%          3.500000
max          5.000000
Name: Rating, dtype: float64


--ESTADISTICAS(RELEASE DATE)
count     60000
unique     8956
top         TBD
freq       8019
Name: Release_Date, dtype: object


#***Limpieza de datos***
Identificamos los valores faltantes, eliminamos duplicados y preparamos los datos para el analisis

In [ ]:
#Funcion para pasar Strings con formato de lista
def recorrer_lista(valor):
  if pd.isna(valor):
    return []
  try:
      resultado = ast.literal_eval(valor)
      if isinstance(resultado, list):
          return [str(x).strip() for x in resultado if str(x).strip()]
      return []
  except (ValueError, SyntaxError):
      return []

#Funcion para convertir las cadenas de 21k a valores numericos
def conversion_numerica(valor):
    if pd.isna(valor):
        return np.nan

    valor = str(valor).strip()

    if valor == "" or valor.lower() == "nan":
        return np.nan
    if valor.endswith("K"):
        return float(valor[:-1]) * 1000
    if valor.endswith("M"):
        return float(valor[:-1]) * 1000000
    return float(valor)

#Funcion para tratar los valores faltantes, eliminar duplicados, y darle formato
def limpieza_datos(df: pd.DataFrame):
  df = df.copy()

  df = df.drop_duplicates()
  if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

  df["Release_Date"]= df["Release_Date"].replace("TBD", np.nan)
  df["Release_Date"]= pd.to_datetime(df["Release_Date"], format='%b %d, %Y', errors='coerce')
  df['Year']= df["Release_Date"].dt.year

  df = df[(df["Year"].isna()) | ((df["Year"] >= 1980) & (df["Year"] <= 2025))]

  df["Summary"] = df["Summary"].fillna("No hay descripcion disponible")

   # --- Columnas de listas (Developers / Genres / Platforms) ---
  df["Developers_list"] = df["Developers"].apply(recorrer_lista)
  df["Genres_list"] = df["Genres"].apply(recorrer_lista)
  df["Platforms_list"] = df["Platforms"].apply(recorrer_lista)

  # --- Columnas numéricas de popularidad (usa map, ver sección de funciones) ---
  columnas_popularidad = ["Plays", "Playing", "Backlogs", "Wishlist", "Lists", "Reviews"]
  for col in columnas_popularidad:
      df[f"{col}_num"] = list(map(conversion_numerica, df[col]))

  # --- Rating: se mantiene NaN cuando falta (57% del dataset no tiene rating);
  #     se documenta la decisión: no se imputa para no sesgar el análisis. ---
  df["Rating"] = pd.to_numeric(df["Rating"], errors="coerce")

  # --- Eliminar filas sin ningún dato útil (sin developer y sin género) ---
  df = df[~((df["Developers_list"].apply(len) == 0) & (df["Genres_list"].apply(len) == 0))]

  print(f"Limpieza finalizada. Dataset resultante: {df.shape[0]} filas, {df.shape[1]} columnas.")
  return df

df_limpio = limpieza_datos(df_crudo)
df_limpio.head(4)


# ***Analisis y manipulacion de datos***
Utilizando la libreria de pandas aplicamos funciones para realizar operaciones aritmeticas, renombrar y organizar columnas